# 02 — NNN baseline reproduction

The Phase-0 charter's first deliverable: reproduce the published NNN
(Grichener et al. 2025, ApJS 279, 49) from the shipped trained models + test
sets, through the upstream evaluation pipeline (patched for portability
only). Pass criterion was "within ~2×"; the achieved result was **exact**
(ratio 1.000 on all 108 comparisons). This is the benchmark every Phase-1
claim is measured against.

Exploratory only — the citable ratios are the RESULTS.md 2026-07-08 rows.

**Note:** `repro/nnn/results/` is gitignored (regenerable). If absent, this
notebook prints regeneration instructions and skips its figures.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nbsupport as nbs

nbs.style()

In [ ]:
nbs.provenance_header(
    "02",
    "NNN baseline reproduction",
    "done",  # no checklist row of its own — a Phase-0 charter deliverable, RESULTS-only
    results_rows=[
        "2026-07-08: ours vs shipped AverageLossesNNN.csv — ratio = 1.000 in all 108 comparisons (18 (net,dt) × 6 metrics)",
        "2026-07-08: model handshake — independent state_dict-walk loader vs upstream class, max abs deviation 0.0 (gate ≤ 1e-6)",
        "2026-07-08: Yₑ improvement over approx21 — reproduced 377–651 % (mesa_80) / 277–390 % (mesa_151) vs published 390–660 / 280–400",
        "2026-07-08: per-isotope mean ΔX band — mesa_80 dt=1e-3 90 % of isotopes in [1e-4, 1e-1], median 3.2e-3",
    ],
    data=[
        "repro/nnn/results/ (gitignored; produced by repro/nnn/run_all.sh from Zenodo 14873443)",
        "data/zenodo/.../CreateFigures/Figure4_main_results (shipped published CSVs)",
    ],
    scripts=[
        "repro/nnn/run_all.sh",
        "repro/nnn/compare_published.py",
        "repro/nnn/check_handshake.py",
        "repro/nnn/check_dx_band.py",
    ],
)

## Guard: are the reproduction outputs present?

In [ ]:
# Constants mirror repro/nnn/compare_published.py (single source: that script).
OURS = nbs.REPO / "repro" / "nnn" / "results"
FIG4 = (
    nbs.REPO / "data" / "zenodo" / "NuclearNeuralNetworks" / "python_scripts_for_analysis"
    / "CreateFigures" / "Figure4_main_results"
)
DTS = ["1e-6", "1e-5", "1e-4", "1e-3", "1e-2", "1e-1", "1e0", "1e1", "1e2"]
IDX = dict(zip(DTS, [134, 184, 234, 284, 334, 384, 434, 484, 534]))
NET_DIR = {"mesa_80": "mesa_80_results", "mesa_151": "mesa151_results"}
LOSS_COLS = [
    "AverageLinearLoss", "AverageYeLoss", "AverageAbarLoss",
    "AverageZbarLoss", "AverageEpsNucLoss", "AverageEpsNuLoss",
]
NETS = list(NET_DIR)

HAVE_REPRO = OURS.is_dir() and os.environ.get("NB_FORCE_NO_REPRO", "0") != "1"
if not HAVE_REPRO:
    print(
        "repro/nnn/results/ not present (it is gitignored — regenerable, not committed).\n"
        "Regenerate with:\n"
        "    bash repro/nnn/run_all.sh\n"
        "The measured verdict stands in RESULTS.md 2026-07-08: ratio ours/theirs = 1.000\n"
        "in all 108 comparisons (worst factor 1.0000); handshake max abs deviation 0.0.\n"
        "Figures below are skipped."
    )
else:
    print(f"found reproduction outputs: {OURS}")

## Figure 1 — ours vs shipped published values (the 108 comparisons)

For every (network, dt) we read OUR `AverageLossesNNN.csv` and THEIR shipped
CSV under `Figure4_main_results/…/timeStepIndex_<idx>/`, and take the ratio
on the last row — 18 (net, dt) × 6 metrics = 108 numbers, every one expected
to be exactly 1.000.

In [ ]:
def last_row(path: Path) -> dict:
    d = pd.read_csv(path)
    return {c: float(d[c].iloc[-1]) for c in d.columns if pd.notna(d[c].iloc[-1])}


if HAVE_REPRO:
    rat = np.full((len(NETS) * len(DTS), len(LOSS_COLS)), np.nan)
    labels = []
    k = 0
    for net in NETS:
        for dt in DTS:
            labels.append(f"{net.replace('mesa_', '')} · {dt}")
            op = OURS / net / dt / "Results" / "Files" / "AverageLossesNNN.csv"
            tp = FIG4 / NET_DIR[net] / dt / f"timeStepIndex_{IDX[dt]}" / "AverageLossesNNN.csv"
            if op.exists() and tp.exists():
                o, t = last_row(op), last_row(tp)
                for j, c in enumerate(LOSS_COLS):
                    if c in o and c in t and t[c] != 0:
                        rat[k, j] = o[c] / t[c]
            k += 1

    finite = rat[np.isfinite(rat)]
    fig, ax = plt.subplots(figsize=(8.5, 8))
    im = ax.imshow(rat, cmap="RdBu_r", vmin=0.5, vmax=1.5, aspect="auto")
    ax.set_xticks(range(len(LOSS_COLS)), [c.replace("Average", "") for c in LOSS_COLS],
                  rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=7)
    for i in range(rat.shape[0]):
        for j in range(rat.shape[1]):
            if np.isfinite(rat[i, j]):
                ax.text(j, i, f"{rat[i, j]:.3f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, label="ratio ours / shipped", shrink=0.6)
    ax.set_title("NNN reproduction: ratio to shipped published values\n"
                 f"({int(np.isfinite(rat).sum())} comparisons; "
                 f"worst |ratio−1| = {np.abs(finite - 1).max():.1e})")
    ax.grid(False)
    nbs.caption(
        fig,
        "Every comparison lands on 1.000 — the reproduction is exact, not merely within the ~2× "
        "Step-2 pass criterion. This validates the whole downstream benchmark: NNN's published "
        "per-dt losses are what our Phase-1 head-to-head will be scored against.",
        results=["RESULTS.md 2026-07-08 NNN-baseline reproduction rows (repro/nnn/, commit de11c8b)"],
        scripts=["repro/nnn/run_all.sh", "repro/nnn/compare_published.py"],
    )
else:
    print("skipped (no reproduction outputs)")

## Figure 2 — per-isotope error distribution vs the paper's Fig. 3 band

`NNNcomps.npz` holds the per-sample predicted/true mass fractions
(`allNNNcompsToSave`, shape (samples, 2, n_isotopes)). The paper reports most
isotopes falling in 1e-4 ≲ ΔXᵢ ≲ 1e-1 with light isotopes best; our audit
measured 90 % of isotopes inside that band at mesa_80 dt=1e-3 (median 3.2e-3,
outliers all BELOW the band — i.e. better than advertised).

In [ ]:
if HAVE_REPRO:
    DEMO = ("mesa_80", "1e-3")
    p = OURS / DEMO[0] / DEMO[1] / "Results" / "Files" / "NNNcomps.npz"
    if p.exists():
        with np.load(p) as z:
            comps = z["allNNNcompsToSave"]  # (n_samples, 2, n_isotopes)
        pred, true = comps[:, 0, :], comps[:, 1, :]
        dx = np.abs(pred - true)
        med = np.median(dx, axis=0)
        p90 = np.percentile(dx, 90, axis=0)
        order = np.argsort(med)

        fig, ax = plt.subplots(figsize=(10, 4.6))
        x = np.arange(len(med))
        ax.fill_between(x, np.maximum(med[order], 1e-12), np.maximum(p90[order], 1e-12),
                        alpha=0.3, color="#0072B2", label="median → p90 per isotope")
        ax.plot(x, np.maximum(med[order], 1e-12), lw=1.4, color="#0072B2", label="median |ΔX|")
        ax.axhspan(1e-4, 1e-1, color="#E69F00", alpha=0.18, label="paper Fig. 3 band 1e-4…1e-1")
        ax.set_yscale("log")
        ax.set_xlabel(f"isotope (sorted by median |ΔX|) — {DEMO[0]}, dt = {DEMO[1]} s")
        ax.set_ylabel("|X_pred − X_true|")
        ax.set_title(f"NNN per-isotope error, {comps.shape[0]} test samples")
        ax.legend(fontsize=8)
        infrac = float(((med >= 1e-4) & (med <= 1e-1)).mean())
        print(f"median-|ΔX| inside the paper band: {infrac:.0%} of isotopes; "
              f"overall median {np.median(dx):.2e}")
        nbs.caption(
            fig,
            "Quick-look re-plot of the shipped-model errors at one (net, dt); the audited numbers "
            "(90 % of isotopes in band at mesa_80 dt=1e-3, median 3.2e-3, outliers all below the "
            "band) come from repro/nnn/check_dx_band.py. Isotopes below the band are ones NNN "
            "predicts better than the paper's stated envelope.",
            results=["RESULTS.md 2026-07-08 per-isotope ΔX band rows (published: arXiv:2503.00115 §3, Fig. 3)"],
            scripts=["repro/nnn/check_dx_band.py"],
        )
    else:
        print(f"no NNNcomps.npz at {p}")
else:
    print("skipped (no reproduction outputs)")

## TODO (stubs)

- **Loss vs dt trend** across the nine timesteps, with the ν-loss crossover
  (NNN better than approx21 for dt ≤ 1e-1 s, worse for dt ∈ {1e0, 1e1, 1e2} —
  RESULTS.md 2026-07-08; the paper says "comparable around dt = 0.1 s").
- **mesa_151 comps panel** — verify the `allNNNcompsToSave` array shape for
  the 151-species net before plotting (mesa_80's shape is read from the file
  above; it is an upstream artifact layout, not a RESULTS.md quantity).

## What this notebook does NOT show

- The improvement-% figures vs approx21 (377–651 % Yₑ etc.): those use the
  shipped `AverageLosses21to80.csv` small-net losses —
  `repro/nnn/compare_published.py` prints the full table.
- Anything about OUR model: Phase-1 work (notebook 11's handoff board).